# Instella Reasoning — GPU Experiment Suite

Runs the GPU half of the experiment plan (see `experiments/README.md`): **generate → score → atlas** for 4 models × 6 benchmarks, then A5 scale + A6 RL emergence. Artifacts land in `experiments/runs/<date>_gpu-suite/`.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Secrets panel (🔑): add `github` (GitHub PAT with read access to this private repo) and `HF_TOKEN` (HuggingFace read token), both with *Notebook access* ON.

Run cells top to bottom. The sanity pass (~45–90 min) proves the pipeline before the real run (~5–10 h).

In [ ]:
# 1) GPU check — fails fast if the runtime is not a GPU
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then restart."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2) Secrets -> environment (token stays out of printed output)
import os
from google.colab import userdata
os.environ["GITHUB_TOKEN"] = userdata.get("github")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HF_HUB_DISABLE_XET"] = "1"
print("secrets loaded")

In [ ]:
# 3) Clone (the experiment branch) + install
import os
if not os.path.isdir("Instella-Reasoning"):
    !git clone -b agent/cpu-experiment-suite https://$GITHUB_TOKEN@github.com/GIND123/Instella-Reasoning
%cd Instella-Reasoning
!pip install -q -e ".[hf,retrieval,viz,stats]"
!pip install -q "bitsandbytes>=0.46.1"
!instella-reasoning --help | head -5

In [ ]:
# 4) SANITY PASS — 20 items per benchmark (~45-90 min). Verify before the real run:
#    every '== generate <model> / <bench> ==' completes, no degenerate-generation warnings.
!bash experiments/run_gpu_suite.sh 20

In [ ]:
# 5) REAL SMOKE RUN — 200 items per benchmark (~5-10 h on a T4).
#    If the session may not last, trim MODELS in experiments/run_gpu_suite.sh and run
#    one model per session — outputs are independent files, partial progress is kept.
!bash experiments/run_gpu_suite.sh 200

In [ ]:
# 6) Quick look: accuracy per (model, benchmark) score file
import json, glob
for f in sorted(glob.glob("experiments/runs/*_gpu-suite/scores/*__*.jsonl")):
    if f.endswith("__ALL.jsonl"):
        continue
    rows = [json.loads(l) for l in open(f) if l.strip()]
    if rows:
        acc = sum(r["correct"] for r in rows) / len(rows)
        print(f"{f.split('/')[-1]:45} {len(rows):4} items  acc={acc:.3f}")

In [ ]:
# 7) Package results and download — unzip into the repo so the folder lands at
#    experiments/runs/<date>_gpu-suite/, then hand it to the analysis step.
!zip -qr gpu_results.zip experiments/runs/*_gpu-suite
from google.colab import files
files.download("gpu_results.zip")